# 02. AI application eval과 error analysis

목표: 단일 accuracy를 넘어 failure category별 지표를 계산하고 다음 개선의 우선순위를 evidence로 정합니다. 외부 model 호출 없이 고정 fixture를 사용합니다.

In [ ]:
examples = [
    {"id": 1, "expected": "refund", "predicted": "refund", "grounded": True, "latency_ms": 420},
    {"id": 2, "expected": "shipping", "predicted": "refund", "grounded": False, "latency_ms": 510},
    {"id": 3, "expected": "account", "predicted": "account", "grounded": False, "latency_ms": 480},
    {"id": 4, "expected": "shipping", "predicted": "shipping", "grounded": True, "latency_ms": 900},
    {"id": 5, "expected": "refund", "predicted": "refund", "grounded": True, "latency_ms": 450},
    {"id": 6, "expected": "account", "predicted": "shipping", "grounded": True, "latency_ms": 430},
]

for item in examples:
    item["correct"] = item["expected"] == item["predicted"]
    if not item["correct"]:
        item["error"] = "intent_classification"
    elif not item["grounded"]:
        item["error"] = "ungrounded_answer"
    elif item["latency_ms"] > 750:
        item["error"] = "latency_slo"
    else:
        item["error"] = "none"

In [ ]:
from collections import Counter

errors = Counter(item["error"] for item in examples)
metrics = {
    "task_accuracy": sum(item["correct"] for item in examples) / len(examples),
    "grounded_rate": sum(item["grounded"] for item in examples) / len(examples),
    "latency_slo_rate": sum(item["latency_ms"] <= 750 for item in examples) / len(examples),
}
print(metrics)
print(errors)

In [ ]:
remedies = {
    "intent_classification": "ambiguous intent fixture를 늘리고 router prompt/label을 비교한다",
    "ungrounded_answer": "retrieval coverage와 citation requirement를 분리 평가한다",
    "latency_slo": "retrieval/model/serialization latency를 trace로 분해한다",
}

backlog = [
    (count, category, remedies[category])
    for category, count in errors.items()
    if category != "none"
]
for count, category, remedy in sorted(backlog, reverse=True):
    print(f"[{count} cases] {category}: {remedy}")

## 다음 loop

가장 빈번하거나 피해가 큰 category 하나만 변경하고 같은 fixture와 새로운 holdout에서 다시 측정하세요. test set을 보면서 prompt를 반복 수정하면 overfitting되므로 development set과 holdout을 분리합니다.